## Phase 0 — Already Done

- data collection
  - NERRS (1995–2025, most regions minus hudson river and 2x great lakes, all stations)
    - originally 15min resolution
    - mostly intact water conditions
    - very sparse nutrient data
      - nutrient data extended ("as-of" forward-fill, 7-day cap)
      - this just in case a record at 945am was skipped over by picking the 1000am record instead to collate at 1hr resolution
    - entirely blank meteorlogical data?
  - ERA5 atmospherics
    - get that atmospheric data back
    - 1hr resolution
    - 0.25 degree resolution
      - not a perfect match to each station in nerrs
      - median centroid was calculated for all stations in a region
      - though most times that centroid ended up nearby but over land (so switched to skin temp instead of water surface temp)
- data cleaning and collation
- split into two datasets
  - full (with nutrients, less complete timeline)
  - and core (without nutrients, more complete)
  - script can be rerun for 1hr resolution
  - but 12hr resolution is included in this repo (zipped)
- various metrics analysis, exploration

## Logic

### Modular Chain
- Model A: ERA5 drivers -> water_temp
- Model B: ERA5 drivers + predicted water_temp -> water properties (salinity, oxygen, pH, etc.)
- Model C: ERA5 drivers + predicted water state -> nutrients (only where nutrient data exists)
  - C can be sacrificed if we run short on time, but it would be a shame

### Why?
- Logical process, re: forcing -> state -> chemistry (deltas from local mean)
- The super sparse nutrient data, if we can keep it, won't impact this as badly as it would one gigantic model
- Easier to read
- Data 'leakage' comes up a lot in reading about this, and this should help reduce it

### To consider...
- When training model B, feed **out-of-fold** predictions (deltas, etc, rather than raw original)
- Then again, if we have time, from B (state) to C (nutrients)

In [ ]:
# first... dependencies
import numpy as np                  # for arrays and math
import pandas as pd                 # for dataframes and csv I/O
import matplotlib.pyplot as plt     # basic visualizations
import seaborn as sns               # for quick readable charts
from sklearn.cluster import KMeans      # simple clustering
from sklearn.decomposition import PCA   # quick 2d display of clusters
from sklearn.preprocessing import StandardScaler  # normalize features
from sklearn.metrics import silhouette_score  # cluster quality check

# keep charts easy to read
sns.set_theme( style = 'whitegrid' )

# the files (thee have been collated and cleaned already)
res = 1 # hours - alternatives 4 and 12

# https://www.youtube.com/watch?v=_W7K79FjI58
# mandatory listening while working on this project

In [ ]:
water = pd.read_csv( f'../data/{res}hr/t4d.{res}hr.water.history.csv' )

water[ 'region' ] = water[ 'region' ].astype( str ).str.strip( ).str.lower( )
water[ 'station' ] = water[ 'station' ].astype( str ).str.strip( ).str.lower( )

# HEE is too sparse for this analysis, so drop it globally here
water = water.loc[ water[ 'region' ] != 'hee' ].copy( )

In [ ]:
# lightweight station and region lookup for labels only
station_lookup = pd.read_csv( '../data/reference/nerrs_station_index.csv' )

station_lookup[ 'region' ] = station_lookup[ 'region_code' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station_full' ] = station_lookup[ 'station' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station' ] = station_lookup[ 'station_full' ].str[ -2: ]

station_lookup = station_lookup[ [ 
    'region',
    'station',
    'station_full',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
] ].drop_duplicates( subset = [ 'region', 'station' ] )

region_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'region_name' ] )
    .drop_duplicates( subset = [ 'region' ] )
    .set_index( 'region' )[ 'region_name' ]
    .to_dict( )
)

station_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'station_name' ] )
    .set_index( [ 'region', 'station' ] )[ 'station_name' ]
    .to_dict( )
)

def station_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]

    name = station_name_lookup.get( ( region_key, station_key ) )

    if pd.isna( name ) or name is None or str( name ).strip( ) == '':
        return station_key

    return f'{station_key} - {name}'

lookup_missing = ( 
    water[ [ 'region', 'station' ] ]
    .drop_duplicates( )
    .merge( station_lookup[ [ 'region', 'station', 'station_name' ] ], on = [ 'region', 'station' ], how = 'left' )
    [ 'station_name' ]
    .isna( )
    .sum( )
)

print( f'lookup missing station names: {int( lookup_missing )}' )

## Phase 1 — Characterization & Classification
Goal: understand what we have before modeling anything

- 1.1 compute per-station summary statistics over a defined baseline period (suggest 1995–2005)
  - mean annual water temp, seasonal amplitude, mean salinity, mean DO
- 1.2 cluster stations in (mean temp × seasonal amplitude) space 
  - k-means, try k=3 or 4, use elbow/silhouette to let the data suggest the right number of groups
  - example in nutrient analysis work
- 1.3 enrich clusters with any available station metadata (estuary type, distance from mouth, watershed area)
  - confirm clusters are physically meaningful, not just statistical artifacts
- 1.4 assign each station a baseline regime label
  - see if kmeans self-identify... 
  - otherwise probably get a rolling means (temp) per station for 1995-2000 to classify
- 1.5 visualize stations on a map colored by regime
  - sanity check that PR/HI are warm, alaska/maine are cold, etc.
- 1.6 document baseline period statistics per regime as a reference table
  - this will be needed for the paper/poster later, too

In [ ]:
# 1.0 - a description
water.describe( ).round( 3 ).T

In [ ]:
# lets make this more readable
water = water.rename( columns = { 
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxy_saturation',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temp_c': 'air_temp'
} )

In [ ]:
# 1.1 - station character baseline (first valid years, not fixed 1995-2000)
# this keeps the idea simple: each station gets its own early baseline window

base_all = water.copy( )
base_all[ 'datetime' ] = pd.to_datetime( base_all[ 'datetime' ], errors = 'coerce' )

base_all = base_all.loc[ 
    :,
    [ 
        'region',
        'station',
        'datetime',
        'water_temp',
        'salinity',
        'oxygen',
        'oxy_saturation',
        'ph',
        'depth',
    ],
].dropna( subset = [ 'datetime' ] )

base_all[ 'year' ] = base_all[ 'datetime' ].dt.year

# annual coverage check so thin years do not define the baseline
year_obs = ( 
    base_all
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

# leap years and all that jazz
year_obs[ 'expected_obs' ] = np.where( 
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70

valid_years = year_obs.loc[ year_obs[ 'year_is_valid' ] ].copy( )
valid_years = valid_years.sort_values( [ 'region', 'station', 'year' ] ).reset_index( drop = True )
valid_years[ 'valid_year_rank' ] = valid_years.groupby( [ 'region', 'station' ] ).cumcount( ) + 1

# first 5 valid years per station
baseline_years = valid_years.loc[ valid_years[ 'valid_year_rank' ] <= 5 ].copy( )

baseline_window = ( 
    baseline_years
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        baseline_start_year = ( 'year', 'min' ),
        baseline_end_year = ( 'year', 'max' ),
        n_valid_years = ( 'year', 'size' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

baseline_window[ 'is_partial_baseline' ] = baseline_window[ 'n_valid_years' ] < 5

# simple rule: keep stations with at least 3 valid years
eligible_stations = baseline_window.loc[ 
    baseline_window[ 'n_valid_years' ] >= 3,
    [ 'region', 'station' ],
]

# pull only rows from each station's selected baseline years
base = base_all.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)

# then keep only stations that passed the >=3-year minimum
base = base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

In [ ]:
# okay, now build station character features from the selected baseline window
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = ( 
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

annual_means = ( 
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

# seasonal amplitudes from monthly climatology
monthly = ( 
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

# building to a new feature, the 'swing' the mean swing of seasonal properties
seasonal_amp = ( 
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

coverage = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

station_baseline = ( 
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .merge( 
        baseline_window[ [ 
            'region',
            'station',
            'baseline_start_year',
            'baseline_end_year',
            'n_valid_years',
            'mean_year_coverage',
            'is_partial_baseline',
        ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# placeholder flag in case we later add nearest-neighbor feature fallback
station_baseline[ 'character_imputed' ] = False
station_baseline_display = ( 
    station_baseline
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

In [ ]:
# quick baseline summary
n_total_stations = baseline_window[ [ 'region', 'station' ] ].drop_duplicates( ).shape[ 0 ]
n_eligible_stations = eligible_stations.shape[ 0 ]
n_full_five_year = int( ( baseline_window[ 'n_valid_years' ] >= 5 ).sum( ) )

print( f'stations with >=3 valid baseline years: {n_eligible_stations} of {n_total_stations}' )
print( f'stations with full 5-year baseline: {n_full_five_year} of {n_total_stations}' )

baseline_preview = ( 
    baseline_window
    .sort_values( [ 'region', 'station' ] )
    .head( 20 )
)

baseline_preview

In [ ]:
eligible_stations_display = ( 
    eligible_stations
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

eligible_stations_display

In [ ]:
# tall small-multiples: one subplot per region, one line per station

plot_df = water.copy( )

plot_df[ 'datetime' ] = pd.to_datetime( plot_df[ 'datetime' ], errors = 'coerce' )
plot_df = plot_df.dropna( subset = [ 'datetime' ] )

temp_col = 'water_temp' if 'water_temp' in plot_df.columns else 'w_temp_c'

plot_df[ 'year' ] = plot_df[ 'datetime' ].dt.year

annual_station = ( 
    plot_df
    .groupby( [ 'region', 'station', 'year' ], as_index = False )[ temp_col ]
    .mean( )
    .rename( columns = { temp_col: 'mean_annual_temp' } )
)

regions = sorted( annual_station[ 'region' ].dropna( ).unique( ) )
n_regions = len( regions )

fig, axes = plt.subplots( 
    n_regions,
    1,
    figsize = ( 16, max( 3 * n_regions, 10 ) ),
    sharex = True,
    constrained_layout = True,
)

if n_regions == 1:
    axes = [ axes ]

for ax, region in zip( axes, regions ):
    sub = annual_station.loc[ annual_station[ 'region' ] == region ].sort_values( [ 'station', 'year' ] )

    for station, g in sub.groupby( 'station' ):
        ax.plot( 
            g[ 'year' ],
            g[ 'mean_annual_temp' ],
            linewidth = 1.4,
            alpha = 0.85,
            label = station_label( region, station ),
        )

    region_title = region_name_lookup.get( region, region )
    ax.set_title( f'{region.upper( )} ( {region_title} ): Mean Annual Water Temp by Station' )
    ax.set_ylabel( 'Temp ( C )' )

    # keep legends readable
    n_stations = sub[ 'station' ].nunique( )

    ax.legend( 
        ncol = 1,
        fontsize = 7,
        frameon = False,
        loc = 'upper left',
        bbox_to_anchor = ( 1.01, 1.0 ),
        borderaxespad = 0,
    )

axes[ -1 ].set_xlabel( 'Year' )
plt.show( )

### 1.2 - Domain-Driven KMeans (Temperature, Salinity, Oxygen, Depth)

single slim clustering pass with interpretable station-character features


In [ ]:
# single, slim KMeans pass with a domain-driven feature subset
# selected features:
# - mean and seasonal amplitude for temperature
# - mean and seasonal amplitude for salinity
# - mean and seasonal amplitude for oxygen
# - mean annual depth

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

domain_feature_cols = [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
]

print( 'domain features:', domain_feature_cols )

domain_input = station_baseline[ [ 'region', 'station' ] + domain_feature_cols ].copy( )

# simple missing-value strategy: region median, then global median
for col in domain_feature_cols:
    domain_input[ col ] = domain_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    domain_input[ col ] = domain_input[ col ].fillna( domain_input[ col ].median( ) )

scaler_domain = StandardScaler( )
X_scaled_domain = scaler_domain.fit_transform( domain_input[ domain_feature_cols ] )

# k scan for elbow + silhouette
k_scan_rows = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_domain )

    k_scan_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_domain, labels_scan ) ),
    } )

k_scan_domain = pd.DataFrame( k_scan_rows )
k_scan_domain[ 'inertia_drop' ] = k_scan_domain[ 'inertia' ].shift( 1 ) - k_scan_domain[ 'inertia' ]
k_scan_domain[ 'inertia_drop_pct' ] = k_scan_domain[ 'inertia_drop' ] / k_scan_domain[ 'inertia' ].shift( 1 )

recommended_k = int( k_scan_domain.loc[ k_scan_domain[ 'silhouette' ].idxmax( ), 'k' ] )

# keep the final pick interpretable for section 1.2
k_min_interpretable = 3
k_max_interpretable = 6
k_scan_interpretable = k_scan_domain.loc[ k_scan_domain[ 'k' ].between( k_min_interpretable, k_max_interpretable ) ].copy( )

if k_scan_interpretable.empty:
    recommended_k_interpretable = 4

else:
    recommended_k_interpretable = int( 
        k_scan_interpretable.loc[ k_scan_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
    )

print( )
print( 'domain-feature k scan:' )
print( k_scan_domain.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k }' )
print( f'interpretable-window K ( { k_min_interpretable }-{ k_max_interpretable } ): { recommended_k_interpretable }' )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan_domain[ 'k' ], k_scan_domain[ 'inertia' ], marker = 'o' )
plt.title( 'Domain Features: K vs Inertia' )
plt.xlabel( 'K' )
plt.ylabel( 'Inertia' )
plt.tight_layout( )
plt.show( )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan_domain[ 'k' ], k_scan_domain[ 'silhouette' ], marker = 'o' )
plt.title( 'Domain Features: K vs Silhouette Score' )
plt.xlabel( 'K' )
plt.ylabel( 'Silhouette' )
plt.tight_layout( )
plt.show( )

# use the bounded recommendation by default; override manually if needed
k_clusters = recommended_k_interpretable
kmeans_model = KMeans( n_clusters = k_clusters, random_state = 42, n_init = 20 )
domain_input[ 'cluster' ] = kmeans_model.fit_predict( X_scaled_domain )

# keep this alias so downstream sections can still use the same naming
domain_input[ 'cluster_domain' ] = domain_input[ 'cluster' ]

domain_silhouette = float( silhouette_score( X_scaled_domain, domain_input[ 'cluster' ] ) )

print( )
print( f'chosen K: { k_clusters }' )
print( 'domain-feature silhouette:', round( domain_silhouette, 4 ) )

print( )
print( 'cluster sizes:' )
print( domain_input[ 'cluster' ].value_counts( ).sort_index( ) )

cluster_profile_raw = domain_input.groupby( 'cluster' )[ domain_feature_cols ].mean( )
cluster_profile = cluster_profile_raw.round( 3 )

cluster_profile_mean = cluster_profile_raw.mean( )
cluster_profile_std = cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
cluster_profile_z = ( cluster_profile_raw - cluster_profile_mean ) / cluster_profile_std


def bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_cluster_name( z_row ):
    temp_tag = bucket_tag( z_row[ 'mean_annual_water_temp' ], 'Cool', 'Temperate', 'Warm' )
    sal_tag = bucket_tag( z_row[ 'mean_annual_salinity' ], 'Fresher', 'Brackish', 'Saline' )
    oxy_tag = bucket_tag( z_row[ 'mean_annual_oxygen' ], 'Low O2', 'Mid O2', 'High O2' )

    amp_tag = bucket_tag( z_row[ 'seasonal_amp_water_temp' ], 'Stable', 'Mixed Seasonality', 'Seasonal' )
    depth_tag = bucket_tag( z_row[ 'mean_annual_depth' ], 'Shallow', 'Mid Depth', 'Deep' )

    name = f'{ temp_tag } / { sal_tag } / { oxy_tag }'
    note = f'{ amp_tag }, { depth_tag }'

    return pd.Series( { 'cluster_name': name, 'cluster_note': note } )


cluster_name_map = cluster_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

print( )
print( 'cluster names:' )
print( cluster_name_map )

domain_input = domain_input.merge( cluster_name_map, on = 'cluster', how = 'left' )

# write cluster labels back to the station tables
for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_note' ]:
    if cluster_col in station_baseline.columns:
        station_baseline = station_baseline.drop( columns = [ cluster_col ] )

station_baseline = station_baseline.merge( 
    domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_note' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

if 'station_baseline_display' in globals( ):
    for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_note' ]:
        if cluster_col in station_baseline_display.columns:
            station_baseline_display = station_baseline_display.drop( columns = [ cluster_col ] )

    station_baseline_display = station_baseline_display.merge( 
        domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_note' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )

# cluster mean feature profiles
plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Cluster Mean Station-Character Features ( Domain Set )' )
plt.xlabel( 'Features' )
plt.ylabel( 'Cluster' )
plt.tight_layout( )
plt.show( )

# correlation matrix for the domain feature set
feature_corr_domain = domain_input[ domain_feature_cols ].corr( )
tri_mask = np.triu( np.ones_like( feature_corr_domain, dtype = bool ), k = 1 )

plt.figure( figsize = ( 10, 8 ) )
sns.heatmap( 
    feature_corr_domain,
    mask = tri_mask,
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm',
    center = 0,
    vmin = -1,
    vmax = 1,
    square = True,
    linewidths = 0.35,
    annot_kws = { 'size': 8 },
    cbar_kws = { 'label': 'Pearson r', 'shrink': 0.85 },
)
plt.title( 'Domain Feature Correlation Matrix ( Triangle + Labels )' )
plt.tight_layout( )
plt.show( )

corr_pairs_domain = ( 
    feature_corr_domain
    .where( np.triu( np.ones( feature_corr_domain.shape ), k = 1 ).astype( bool ) )
    .stack( )
    .reset_index( )
    .rename( columns = { 'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'corr' } )
)

corr_pairs_domain[ 'abs_corr' ] = corr_pairs_domain[ 'corr' ].abs( )
corr_pairs_domain = corr_pairs_domain.sort_values( 'abs_corr', ascending = False )

print( )
print( 'top correlated domain-feature pairs:' )
corr_pairs_domain.head( 12 )

# PCA scatter of resulting clusters
pca_domain = PCA( n_components = 2 )
pcs_domain = pca_domain.fit_transform( X_scaled_domain )

pca_loadings_domain = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ 'PC1', 'PC2' ],
)

# for eventual readability 
feature_phrase_map = { 
    'mean_annual_water_temp': ( 'warmer mean temp', 'cooler mean temp' ),
    'seasonal_amp_water_temp': ( 'larger temp seasonality', 'smaller temp seasonality' ),
    'mean_annual_salinity': ( 'saltier water', 'fresher water' ),
    'seasonal_amp_salinity': ( 'larger salinity swings', 'smaller salinity swings' ),
    'mean_annual_oxygen': ( 'higher mean oxygen', 'lower mean oxygen' ),
    'seasonal_amp_oxygen': ( 'larger oxygen swings', 'smaller oxygen swings' ),
    'mean_annual_depth': ( 'deeper stations', 'shallower stations' ),
}


# automate, if at all possible, interpreting the PCAs
def pc_axis_interpretation( component_name ):
    top_features = pca_loadings_domain[ component_name ].abs( ).sort_values( ascending = False ).head( 2 ).index
    text_parts = [ ]

    for feat in top_features:
        pos_text, neg_text = feature_phrase_map[ feat ]
        loading = pca_loadings_domain.loc[ feat, component_name ]
        text_parts.append( pos_text if loading >= 0 else neg_text )

    return ' + '.join( text_parts )


pc1_var_pct = round( float( pca_domain.explained_variance_ratio_[ 0 ] ) * 100, 1 )
pc2_var_pct = round( float( pca_domain.explained_variance_ratio_[ 1 ] ) * 100, 1 )
pc1_label_text = pc_axis_interpretation( 'PC1' )
pc2_label_text = pc_axis_interpretation( 'PC2' )

domain_plot = domain_input[ [ 'region', 'station', 'cluster', 'cluster_name', 'cluster_note' ] ].copy( )
domain_plot[ 'pc1' ] = pcs_domain[ :, 0 ]
domain_plot[ 'pc2' ] = pcs_domain[ :, 1 ]

cluster_centers = ( 
    domain_plot
    .groupby( [ 'cluster', 'cluster_name', 'cluster_note' ], as_index = False )[ [ 'pc1', 'pc2' ] ]
    .mean( )
)

plt.figure( figsize = ( 12, 8 ) )
sns.scatterplot( 
    data = domain_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster',
    palette = 'tab10',
    s = 70,
    alpha = 0.88,
)

for _, row in cluster_centers.iterrows( ):
    label_text = '\n'.join( [ str( row[ 'cluster_name' ] ), str( row[ 'cluster_note' ] ) ] )

    plt.text( 
        row[ 'pc1' ],
        row[ 'pc2' ],
        label_text,
        ha = 'center',
        va = 'center',
        fontsize = 8,
        weight = 'bold',
        bbox = { 'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'alpha': 0.72, 'edgecolor': 'gray' },
    )

plt.title( 'Domain-Feature KMeans Clusters ( PCA View + Named Cluster Centers )' )
plt.xlabel( f'PC1 ( { pc1_var_pct }% var ): { pc1_label_text }' )
plt.ylabel( f'PC2 ( { pc2_var_pct }% var ): { pc2_label_text }' )
plt.tight_layout( )
plt.show( )

if 'station_baseline_display' in globals( ):
    station_baseline_display[ [ 
        'region',
        'station',
        'station_name',
        'cluster',
        'cluster_name',
        'cluster_note',
        'baseline_start_year',
        'baseline_end_year',
        'n_valid_years',
    ] ].sort_values( [ 'cluster', 'region', 'station' ] ).head( 40 )

else:
    station_baseline[ [ 
        'region',
        'station',
        'cluster',
        'cluster_name',
        'cluster_note',
        'baseline_start_year',
        'baseline_end_year',
        'n_valid_years',
    ] ].sort_values( [ 'cluster', 'region', 'station' ] ).head( 40 )



#### PCA Interpretation (Domain Feature Set)

- **PC1** and **PC2** are weighted blends of the selected domain features.
- Use **explained variance ratio** to see how much structure each component captures.
- Use **absolute loadings** to identify which features dominate each component.
- Nearby points in PCA space still represent similar station character.


In [ ]:
# quick PCA interpretation table for the domain feature run
pca_variance = pd.Series( 
    pca_domain.explained_variance_ratio_,
    index = [ f'PC{i+1}' for i in range( len( pca_domain.explained_variance_ratio_ ) ) ],
    name = 'explained_variance_ratio',
)

pca_loadings = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ f'PC{i+1}' for i in range( pca_domain.n_components_ ) ],
)

print( 'explained variance ratio:' )
print( pca_variance.round( 3 ) )

pca_loadings.round( 3 )

### 1.3 - Cluster Context and Distribution

prepare cluster-station context and inspect distribution by region


In [ ]:
# 1.3 - cluster context check with available station metadata
# keep a clean station-level table for context and distribution checks

if 'station_baseline_display' in globals( ):
    cluster_station = station_baseline_display.copy( )

else:
    cluster_station = station_baseline.copy( )

    cluster_station = cluster_station.merge( 
        station_lookup[ [ 'region', 'station', 'station_name', 'region_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )

# fallback labels in case this cell is run before 1.2 naming is created
if 'cluster_name' not in cluster_station.columns:
    cluster_station[ 'cluster_name' ] = 'Cluster ' + cluster_station[ 'cluster' ].astype( str )

if 'cluster_note' not in cluster_station.columns:
    cluster_station[ 'cluster_note' ] = ''

cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

cluster_distribution_region = pd.crosstab( 
    cluster_station[ 'region' ],
    cluster_station[ 'cluster' ],
)

print( 'cluster distribution by region:' )
cluster_distribution_region

plt.figure( figsize = ( 10, 6 ) )
sns.heatmap( cluster_distribution_region, annot = True, fmt = 'd', cmap = 'Blues' )
plt.title( 'Cluster Distribution by Region' )
plt.xlabel( 'Cluster' )
plt.ylabel( 'Region' )
plt.tight_layout( )
plt.show( )

### 1.4 - Assign Regime Labels to the Working Dataset (Slim Join)

add only the cluster id to the large working table, and keep descriptive labels in small lookup tables


In [ ]:
# 1.4 - slim cluster assignment to the large working table
# keep it lean: only add cluster id to water, keep descriptive labels in small lookup tables

station_cluster_lookup = ( 
    station_baseline[ [ 'region', 'station', 'cluster' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

cluster_label_lookup = ( 
    station_baseline[ [ 'cluster', 'cluster_name', 'cluster_note' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# save the small lookup tables for reproducibility
station_cluster_lookup_path = '../data/reference/t4d.station_cluster.lookup.csv'
cluster_label_lookup_path = '../data/reference/t4d.cluster_label.lookup.csv'

station_cluster_lookup.to_csv( station_cluster_lookup_path, index = False )
cluster_label_lookup.to_csv( cluster_label_lookup_path, index = False )

# add only cluster id to the large water table
if 'cluster' in water.columns:
    water = water.drop( columns = [ 'cluster' ] )

water = water.merge( station_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

n_rows = len( water )
n_clustered = int( water[ 'cluster' ].notna( ).sum( ) )
cluster_coverage = round( 100 * n_clustered / n_rows, 2 ) if n_rows > 0 else np.nan

water_memory_mb = round( water.memory_usage( deep = True ).sum( ) / 1024 / 1024, 2 )

print( f'water rows with cluster assigned: {n_clustered} / {n_rows} ( {cluster_coverage}% )' )
print( f'current water table size (MB): {water_memory_mb}' )
print( f'saved: {station_cluster_lookup_path}' )
print( f'saved: {cluster_label_lookup_path}' )

print( )
print( 'cluster label lookup:' )
cluster_label_lookup

### 1.5 - Map Stations by Regime

plot station locations colored by cluster, with rough land/ocean contours when available


In [ ]:
# 1.5 - map stations by cluster (static matplotlib)
from pathlib import Path
import geopandas as gpd

if 'cluster_station' not in globals( ):
    if 'station_baseline_display' in globals( ):
        cluster_station = station_baseline_display.copy( )

    else:
        cluster_station = station_baseline.copy( )

        cluster_station = cluster_station.merge( 
            station_lookup[ [ 'region', 'station', 'station_name', 'region_name', 'latitude', 'longitude' ] ],
            on = [ 'region', 'station' ],
            how = 'left',
        )

    if 'cluster_name' not in cluster_station.columns:
        cluster_station[ 'cluster_name' ] = 'Cluster ' + cluster_station[ 'cluster' ].astype( str )

cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

map_df = cluster_station.dropna( subset = [ 'latitude', 'longitude', 'cluster' ] ).copy( )

cluster_ids = sorted( map_df[ 'cluster' ].dropna( ).astype( int ).unique( ) )
palette = sns.color_palette( 'tab10', n_colors = max( len( cluster_ids ), 1 ) )
cluster_color_map = { cid: palette[ idx % len( palette ) ] for idx, cid in enumerate( cluster_ids ) }

center_df = ( 
    map_df
    .groupby( [ 'cluster', 'cluster_name' ], as_index = False )[ [ 'longitude', 'latitude' ] ]
    .mean( )
)

lon_min = map_df[ 'longitude' ].min( )
lon_max = map_df[ 'longitude' ].max( )
lat_min = map_df[ 'latitude' ].min( )
lat_max = map_df[ 'latitude' ].max( )

lon_pad = max( 1.5, ( lon_max - lon_min ) * 0.12 )
lat_pad = max( 1.0, ( lat_max - lat_min ) * 0.12 )

fig, ax = plt.subplots( figsize = ( 12, 7 ) )
ax.set_facecolor( '#d7ecff' )

# find a local Natural Earth shapefile path robustly
ne_path = None

try:
    import pyogrio

    candidate = Path( pyogrio.__file__ ).resolve( ).parent / 'tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp'

    if candidate.exists( ):
        ne_path = candidate

except Exception:
    ne_path = None

if ne_path is None:
    search_roots = [ Path.cwd( ), Path.cwd( ).parent, Path.cwd( ).parent.parent ]

    for root in search_roots:
        hits = list( root.glob( '.venv/lib/python*/site-packages/pyogrio/tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp' ) )

        if len( hits ) > 0:
            ne_path = hits[ 0 ]
            break

if ne_path is not None and ne_path.exists( ):
    world = gpd.read_file( ne_path )

    world.plot( 
        ax = ax,
        color = '#f2efe4',
        edgecolor = 'none',
        zorder = 0,
    )

    world.boundary.plot( 
        ax = ax,
        color = '#666666',
        linewidth = 0.45,
        zorder = 1,
    )

else:
    print( 'coastline layer not found; drawing points only' )

for cid in cluster_ids:
    sub = map_df.loc[ map_df[ 'cluster' ] == cid ]

    ax.scatter( 
        sub[ 'longitude' ],
        sub[ 'latitude' ],
        s = 70,
        alpha = 0.88,
        color = cluster_color_map[ cid ],
        label = f'Cluster {cid}',
        zorder = 2,
    )

for _, row in center_df.iterrows( ):
    ax.text( 
        row[ 'longitude' ],
        row[ 'latitude' ],
        str( row[ 'cluster_name' ] ),
        fontsize = 7,
        ha = 'center',
        va = 'center',
        bbox = { 'boxstyle': 'round,pad=0.25', 'facecolor': 'white', 'alpha': 0.70, 'edgecolor': 'gray' },
        zorder = 3,
    )

ax.set_xlim( lon_min - lon_pad, lon_max + lon_pad )
ax.set_ylim( lat_min - lat_pad, lat_max + lat_pad )
ax.set_xlabel( 'Longitude' )
ax.set_ylabel( 'Latitude' )
ax.set_title( 'Station Map by Cluster ( Static Map + Coastlines )' )
ax.legend( title = 'Cluster', loc = 'lower left', frameon = True )

plt.tight_layout( )
plt.show( )


### 1.6 - Reference Tables for Reporting

commit cluster summary and station tables to small reference objects/files


In [ ]:
# 1.6 - save reference tables for reporting
if 'cluster_station' not in globals( ):
    if 'station_baseline_display' in globals( ):
        cluster_station = station_baseline_display.copy( )

    else:
        cluster_station = station_baseline.copy( )

        cluster_station = cluster_station.merge( 
            station_lookup[ [ 'region', 'station', 'station_name', 'region_name', 'latitude', 'longitude' ] ],
            on = [ 'region', 'station' ],
            how = 'left',
        )

    if 'cluster_name' not in cluster_station.columns:
        cluster_station[ 'cluster_name' ] = 'Cluster ' + cluster_station[ 'cluster' ].astype( str )

    if 'cluster_note' not in cluster_station.columns:
        cluster_station[ 'cluster_note' ] = ''

cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

cluster_specs = ( 
    cluster_station
    .groupby( 'cluster', as_index = False )
    .agg( # gather the following specs for each station cluster: (see kmeans earlier)
        cluster_name = ( 'cluster_name', 'first' ),
        cluster_note = ( 'cluster_note', 'first' ),
        n_stations = ( 'station', 'nunique' ),
        n_regions = ( 'region', 'nunique' ),
        mean_annual_water_temp = ( 'mean_annual_water_temp', 'mean' ),
        seasonal_amp_water_temp = ( 'seasonal_amp_water_temp', 'mean' ),
        mean_annual_salinity = ( 'mean_annual_salinity', 'mean' ),
        seasonal_amp_salinity = ( 'seasonal_amp_salinity', 'mean' ),
        mean_annual_oxygen = ( 'mean_annual_oxygen', 'mean' ),
        seasonal_amp_oxygen = ( 'seasonal_amp_oxygen', 'mean' ),
        mean_annual_depth = ( 'mean_annual_depth', 'mean' ),
        mean_latitude = ( 'latitude', 'mean' ),
        mean_longitude = ( 'longitude', 'mean' ),
    )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# a tad more readable ... 
for col in [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
    'mean_latitude',
    'mean_longitude',
]:
    cluster_specs[ col ] = cluster_specs[ col ].round( 3 )

cluster_station_table = cluster_station[ [ 
    'region',
    'region_name',
    'station',
    'station_name',
    'latitude',
    'longitude',
    'cluster',
    'cluster_name',
    'cluster_note',
] ].sort_values( [ 'cluster', 'region', 'station' ] )

# caching the cluster specifications as a reference guide later
# if we reconfigure anyhting about the k-means work, will have to delete these
cluster_specs_path = '../data/reference/t4d.domain.cluster.specs.csv'
cluster_station_table_path = '../data/reference/t4d.cluster.station.table.csv'

cluster_specs.to_csv( cluster_specs_path, index = False )
cluster_station_table.to_csv( cluster_station_table_path, index = False )

#print( f'saved: {cluster_specs_path}' )
#print( f'saved: {cluster_station_table_path}' )

print( 'station table sample:' )
cluster_station_table.head( 80 )

In [ ]:
# phase 1 cleanup ... keep only reference tables needed later
# otherwise the vars list is just... hard to read
phase1_temp_vars = [ 
    'base_all',
    'year_obs',
    'valid_years',
    'baseline_years',
    'baseline_window',
    'eligible_stations',
    'base',
    'annual',
    'annual_means',
    'monthly',
    'seasonal_amp',
    'depth_stats',
    'coverage',
    'station_baseline_display',
    'n_total_stations',
    'n_eligible_stations',
    'n_full_five_year',
    'baseline_preview',
    'eligible_stations_display',
    'plot_df',
    'temp_col',
    'annual_station',
    'regions',
    'n_regions',
    'domain_feature_cols',
    'domain_input',
    'scaler_domain',
    'X_scaled_domain',
    'k_scan_rows',
    'k_scan_domain',
    'recommended_k',
    'k_min_interpretable',
    'k_max_interpretable',
    'k_scan_interpretable',
    'recommended_k_interpretable',
    'k_clusters',
    'kmeans_model',
    'domain_silhouette',
    'cluster_profile_raw',
    'cluster_profile',
    'cluster_profile_mean',
    'cluster_profile_std',
    'cluster_profile_z',
    'cluster_name_map',
    'feature_corr_domain',
    'tri_mask',
    'corr_pairs_domain',
    'pca_domain',
    'pcs_domain',
    'pca_loadings_domain',
    'feature_phrase_map',
    'pc1_var_pct',
    'pc2_var_pct',
    'pc1_label_text',
    'pc2_label_text',
    'domain_plot',
    'cluster_centers',
    'pca_variance',
    'pca_loadings',
    'cluster_distribution_region',
    'cluster_specs_path',
    'cluster_station_table_path',
    'map_df',
    'cluster_ids',
    'palette',
    'cluster_color_map',
    'center_df',
    'lon_min',
    'lon_max',
    'lat_min',
    'lat_max',
    'lon_pad',
    'lat_pad',
    'ne_path',
]

for var_name in phase1_temp_vars:
    if var_name in globals( ):
        del globals( )[ var_name ]


## Phase 2 — Temporal Diagnostics
Goal: characterize lag structure and trends before building predictive features

- 2.1 run STL decomposition on water temperature per station
  - annual + diurnal cycles 
  - extract trend components
  - see trends analysis for example
- 2.2 compute cross-correlations between air temp and water temp across a range of lags (0–28 days)
  - identify lag-at-peak-correlation per station
- 2.3 repeat cross-correlation for wind/precip → salinity, air temp → DO
  - reading suggests DO may be longer lag times
- 2.4 summarize lag structure by regime
  - do cold estuaries respond more slowly than warm ones?
  - other oddities? 
  - stuff we'd report on in paper/poster
- 2.5 identify stations showing trend drift in the STL trend component
  - lag candidates for regime-transition analysis
  - feeds into phase 5, btw...

In [ ]:
# helper function to find best lag correlation for a given driver-response pair
def best_lag_table( daily_df, driver_col, response_col, lag_max = 28, min_pairs = 40 ):
    rows = [ ]

    for ( region, station ), g in daily_df.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )

        driver = s[ driver_col ]
        response = s[ response_col ]

        best_lag = np.nan
        best_corr = np.nan
        best_n = 0

        for lag in range( lag_max + 1 ):
            # lag means driver leads response by `lag` days
            pair = pd.concat( [ response, driver.shift( lag ) ], axis = 1 ).dropna( )

            if len( pair ) < min_pairs:
                continue

            # how strongly do they align?
            corr = pair.iloc[ :, 0 ].corr( pair.iloc[ :, 1 ] )

            if pd.isna( corr ):
                continue

            # hold on to the strong correlation
            if pd.isna( best_corr ) or abs( corr ) > abs( best_corr ):
                best_lag = lag
                best_corr = corr
                best_n = len( pair )

        # save it
        rows.append( { 
            'region': region,
            'station': station,
            'best_lag_days': best_lag,
            'peak_corr': best_corr,
            'n_pairs': best_n,
        } )

    out = pd.DataFrame( rows )

    out = out.merge( 
        daily_df[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
        on = [ 'region', 'station' ],
        how = 'left',
    )

    return out

In [ ]:
# 2.1 - temporal diagnostics (first working pass)
# simple outputs: daily_air diagnostics + STL trend summary + lag-at-peak-correlation tables
from pathlib import Path
from statsmodels.tsa.seasonal import STL

phase2_out_dir = '../data/reference/phase2'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

# this is a bit of work so the results will be cached to file and checked
# for if more runs are done that don't need to redo this
# to redo it... delete the cache files
daily_cache_path = f'{phase2_out_dir}/t4d.phase2.daily.csv'
stl_summary_path = f'{phase2_out_dir}/t4d.phase2.stl.summary.csv'
lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'
lag_cluster_summary_path = f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv'
trend_drift_candidates_path = f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv'

core_cache_paths = [ 
    daily_cache_path,
    stl_summary_path,
    lag_air_to_water_path,
    lag_wind_to_salinity_path,
    lag_precip_to_salinity_path,
    lag_air_to_oxygen_path,
]

# basically.. it took about 10 min per run,
# and that's a long time to wait just to change a font in a chart later...
phase2_cache_loaded = all( [ Path( fp ).exists( ) for fp in core_cache_paths ] )
if phase2_cache_loaded:
    print( 'loading cached phase 2 tables before recompute...' )

    daily_air = pd.read_csv( daily_cache_path )
    daily_air[ 'date' ] = pd.to_datetime( daily_air[ 'date' ], errors = 'coerce' )

    stl_summary = pd.read_csv( stl_summary_path )
    lag_air_to_water = pd.read_csv( lag_air_to_water_path )
    lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
    lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
    lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )

    if Path( lag_cluster_summary_path ).exists( ):
        lag_cluster_summary = pd.read_csv( lag_cluster_summary_path )

    if Path( trend_drift_candidates_path ).exists( ):
        trend_drift_candidates = pd.read_csv( trend_drift_candidates_path )

    print( 'cached row counts:' )
    print( 'daily rows:', len( daily_air ) )
    print( 'stl summary:', len( stl_summary ) )
    print( 'air -> water lag:', len( lag_air_to_water ) )
    print( 'wind -> salinity lag:', len( lag_wind_to_salinity ) )
    print( 'precip -> salinity lag:', len( lag_precip_to_salinity ) )
    print( 'air -> oxygen lag:', len( lag_air_to_oxygen ) )

else:
    print( 'phase 2 cache not complete; computing fresh outputs...' )

    phase2 = water.copy( )
    phase2[ 'datetime' ] = pd.to_datetime( phase2[ 'datetime' ], errors = 'coerce' )
    phase2 = phase2.dropna( subset = [ 'datetime' ] )

    # fix air temperature column naming from raw files if needed
    if 'air_temp' not in phase2.columns:
        if 'm_temp_c' in phase2.columns:
            phase2[ 'air_temp' ] = phase2[ 'm_temp_c' ]

    # make sure we have cluster on each row
    if 'cluster' not in phase2.columns:
        phase2 = phase2.merge( 
            station_baseline[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
            on = [ 'region', 'station' ],
            how = 'left',
        )

    phase2[ 'date' ] = phase2[ 'datetime' ].dt.floor( 'D' )

    daily_air = ( 
        phase2
        .groupby( [ 'region', 'station', 'cluster', 'date' ], as_index = False )
        .agg( 
            water_temp_daily = ( 'water_temp', 'mean' ),
            salinity_daily = ( 'salinity', 'mean' ),
            oxygen_daily = ( 'oxygen', 'mean' ),
            air_temp_daily = ( 'air_temp', 'mean' ),
            wind_speed_daily = ( 'wind_speed', 'mean' ),
            precip_daily = ( 'precipitation', 'sum' ),
        )
    )

    print( 'daily diagnostic rows:', len( daily_air ) )
    daily_air.to_csv( daily_cache_path, index = False )

    # 2.1 STL on daily water temperature (annual cycle)
    stl_rows = [ ]

    for ( region, station ), g in daily_air.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )[ 'water_temp_daily' ]

        # regular daily index so STL has a stable cadence
        full_idx = pd.date_range( s.index.min( ), s.index.max( ), freq = 'D' )
        s = s.reindex( full_idx )

        if s.notna( ).sum( ) < 365:
            continue

        s = s.interpolate( limit_direction = 'both' )

        if s.notna( ).sum( ) < 365:
            continue

        # reminder .. STL is seasonal trend decomposition based on LOESS smoothing
        # it's robust to outliers and can handle some missing data
        # but it does require a regular time index and enough data to identify the seasonal pattern
        stl = STL( s, period = 365, robust = True ).fit( )

        trend = stl.trend
        slope_per_day = np.polyfit( np.arange( len( trend ) ), trend, 1 )[ 0 ]
        slope_per_year = slope_per_day * 365

        stl_rows.append( { 
            'region': region,
            'station': station,
            'cluster': g[ 'cluster' ].dropna( ).iloc[ 0 ] if g[ 'cluster' ].notna( ).any( ) else np.nan,
            'n_days': int( len( s ) ),
            'stl_trend_slope_c_per_year': float( slope_per_year ),
            'stl_seasonal_amp_c': float( stl.seasonal.max( ) - stl.seasonal.min( ) ),
        } )

    stl_summary = pd.DataFrame( stl_rows )
    print( 'stl station count:', len( stl_summary ) )

### 2.2 and 2.3 - build lag tables

In [ ]:
# 2.2 and 2.3 lag tables (with cache)
from pathlib import Path

# don't redo this work if it's done
# but if you WANT to, then you'll have to clean the files out physically and rerun
if 'phase2_cache_loaded' in globals( ) and phase2_cache_loaded:
    print( 'using lag tables already loaded from cache in 2.1' )

else:
    phase2_out_dir = '../data/reference/phase2'
    Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

    lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
    lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
    lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
    lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'

    lag_paths = [ 
        lag_air_to_water_path,
        lag_wind_to_salinity_path,
        lag_precip_to_salinity_path,
        lag_air_to_oxygen_path,
    ]

    if all( [ Path( fp ).exists( ) for fp in lag_paths ] ):
        print( 'loading cached lag tables from disk...' )

        lag_air_to_water = pd.read_csv( lag_air_to_water_path )
        lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
        lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
        lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )

    else:
        print( 'lag cache not found; computing lag tables...' )

        # defined up above
        # rinse and repeat
        lag_air_to_water = best_lag_table( daily_air, 'air_temp_daily', 'water_temp_daily', lag_max = 28 )
        lag_wind_to_salinity = best_lag_table( daily_air, 'wind_speed_daily', 'salinity_daily', lag_max = 28 )
        lag_precip_to_salinity = best_lag_table( daily_air, 'precip_daily', 'salinity_daily', lag_max = 28 )
        lag_air_to_oxygen = best_lag_table( daily_air, 'air_temp_daily', 'oxygen_daily', lag_max = 28 )

        lag_air_to_water.to_csv( lag_air_to_water_path, index = False )
        lag_wind_to_salinity.to_csv( lag_wind_to_salinity_path, index = False )
        lag_precip_to_salinity.to_csv( lag_precip_to_salinity_path, index = False )
        lag_air_to_oxygen.to_csv( lag_air_to_oxygen_path, index = False )

print( )
print( 'lag table row counts:' )
print( 'air -> water:', len( lag_air_to_water ) )
print( 'wind -> salinity:', len( lag_wind_to_salinity ) )
print( 'precip -> salinity:', len( lag_precip_to_salinity ) )
print( 'air -> oxygen:', len( lag_air_to_oxygen ) )

### 2.3b - visualize lag table signals

In [ ]:
lag_tables = { 
    'air -> water': lag_air_to_water,
    'wind -> salinity': lag_wind_to_salinity,
    'precip -> salinity': lag_precip_to_salinity,
    'air -> oxygen': lag_air_to_oxygen,
}

lag_plot_rows = [ ]
for relation, lag_df in lag_tables.items( ):
    lag_tmp = lag_df.copy( )
    lag_tmp[ 'relation' ] = relation
    lag_plot_rows.append( lag_tmp )

lag_plot = pd.concat( lag_plot_rows, ignore_index = True )
lag_plot = lag_plot.dropna( subset = [ 'best_lag_days', 'peak_corr' ] )
lag_plot[ 'best_lag_days' ] = lag_plot[ 'best_lag_days' ].astype( int )

lag_signal_summary = ( 
    lag_plot
    .groupby( 'relation', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        median_best_lag_days = ( 'best_lag_days', 'median' ),
        mean_best_lag_days = ( 'best_lag_days', 'mean' ),
        mean_abs_peak_corr = ( 'peak_corr', lambda s: s.abs( ).mean( ) ),
    )
)

for col in [ 'median_best_lag_days', 'mean_best_lag_days', 'mean_abs_peak_corr' ]:
    lag_signal_summary[ col ] = lag_signal_summary[ col ].round( 3 )

print( 'lag signal summary:' )
display( lag_signal_summary )

fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

sns.histplot( 
    data = lag_plot,
    x = 'best_lag_days',
    hue = 'relation',
    multiple = 'dodge',
    discrete = True,
    shrink = 0.85,
    ax = axes[ 0 ],
)
axes[ 0 ].set_title( 'Best-Lag Distribution by Relationship' )
axes[ 0 ].set_xlabel( 'Best Lag (days)' )
axes[ 0 ].set_ylabel( 'Station Count' )

sns.boxplot( 
    data = lag_plot,
    x = 'relation',
    y = 'peak_corr',
    ax = axes[ 1 ],
)
axes[ 1 ].axhline( 0, color = 'black', linestyle = '--', linewidth = 1 )
axes[ 1 ].set_title( 'Peak Correlation Spread by Relationship' )
axes[ 1 ].set_xlabel( '' )
axes[ 1 ].set_ylabel( 'Peak Correlation' )
axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

plt.tight_layout( )
plt.show( )

lag_count = ( 
    lag_plot
    .groupby( [ 'relation', 'best_lag_days' ] )
    .size( )
    .reset_index( name = 'n_stations' )
)

lag_heat = lag_count.pivot( index = 'relation', columns = 'best_lag_days', values = 'n_stations' ).fillna( 0 )

plt.figure( figsize = ( 12, 4 ) )
sns.heatmap( lag_heat, annot = True, fmt = '.0f', cmap = 'Blues' )
plt.title( 'Station Counts by Best Lag Day and Relationship' )
plt.xlabel( 'Best Lag (days)' )
plt.ylabel( '' )
plt.tight_layout( )
plt.show( )

### 2.3c - reading the lag visuals (what / how / why)

**What this is:**
- compact view of lag behavior across station-level relationships:
  - air -> water temp
  - wind -> salinity
  - precip -> salinity
  - air -> oxygen
- shows (1) where best lags concentrate, and (2) how strong peak correlations are

**How it was built:**
- phase 2 lag tables already loaded/computed above
- stacks them into one tidy frame with a `relation` label
- keeps station-level `best_lag_days` and `peak_corr`.
- produces ...
  - summary table by relation
  - histogram of best lag day counts
  - boxplot of peak correlations
  - heatmap of station counts by lag day

**Why this matters:**
- confirms whether lag behavior is tight or broad by driver-response pair
- helps justify Phase 3 feature choices (fixed windows vs lag-specific features).
- gives interpretable evidence for writeup/poster about system response timing.
- flags whether some links are weaker/noisier (wider correlation spread), which is useful for model expectations in later phases

### 2.4 - summarize lag structure by cluster type

In [ ]:
lag_cluster_summary = ( 
    lag_air_to_water
    .groupby( 'cluster', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        mean_lag_days_air_to_water = ( 'best_lag_days', 'mean' ),
        median_lag_days_air_to_water = ( 'best_lag_days', 'median' ),
        mean_peak_corr_air_to_water = ( 'peak_corr', 'mean' ),
    )
)

for col in [ 'mean_lag_days_air_to_water', 'median_lag_days_air_to_water', 'mean_peak_corr_air_to_water' ]:
    lag_cluster_summary[ col ] = lag_cluster_summary[ col ].round( 3 )

### 2.5 - trend drift candidates?

In [ ]:
# simple threshold: |slope| >= 0.05 c per year
trend_drift_threshold = 0.05
trend_drift = stl_summary.copy( )
trend_drift[ 'abs_trend_slope' ] = trend_drift[ 'stl_trend_slope_c_per_year' ].abs( )
trend_drift_candidates = trend_drift.loc[ trend_drift[ 'abs_trend_slope' ] >= trend_drift_threshold ].copy( )
trend_drift_candidates = trend_drift_candidates.sort_values( 'abs_trend_slope', ascending = False )

# save phase 2 outputs
phase2_out_dir = '../data/reference/phase2'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

daily_air.to_csv( f'{phase2_out_dir}/t4d.phase2.daily.csv', index = False )
stl_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.stl.summary.csv', index = False )
lag_air_to_water.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv', index = False )
lag_wind_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv', index = False )
lag_precip_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv', index = False )
lag_air_to_oxygen.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv', index = False )
lag_cluster_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv', index = False )
trend_drift_candidates.to_csv( f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv', index = False )

print( f'saved phase 2 tables under: {phase2_out_dir}' )


In [ ]:
print( 'lag summary by cluster (air -> water):' )
lag_cluster_summary

In [ ]:
print( 'top trend drift candidates:' )
trend_drift_candidates.head( 30 )

In [ ]:
# let's visualize the lag relationships for the cluster summary
# via a boxplot of the station-level lag values by cluster
plt.figure( figsize = ( 10, 6 ) )
sns.boxplot( data = trend_drift_candidates, x = 'cluster', y = 'stl_trend_slope_c_per_year' )
plt.title( 'Annual Trend (in C)' )
plt.xlabel( 'Cluster' )
plt.ylabel( 'Trend' )
plt.tight_layout( )
plt.show( )

In [ ]:
# let's visualize the lag relationships for the cluster summary
# via a boxplot of the station-level lag values by cluster
plt.figure( figsize = ( 10, 6 ) )
sns.boxplot( data = trend_drift_candidates, x = 'cluster', y = 'stl_seasonal_amp_c' )
plt.title( 'Seasonal Amplitide (in C)' )
plt.xlabel( 'Cluster' )
plt.ylabel( 'Trend' )
plt.tight_layout( )
plt.show( )

In [ ]:
# phase 2 cleanup .. keep phase 2 output tables and daily_air diagnostics
phase2_temp_vars = [ 
    'phase2',
    'stl_rows',
    'trend_drift',
    'trend_drift_threshold',
    'phase2_out_dir',
    'daily_cache_path',
    'stl_summary_path',
    'lag_air_to_water_path',
    'lag_wind_to_salinity_path',
    'lag_precip_to_salinity_path',
    'lag_air_to_oxygen_path',
    'lag_cluster_summary_path',
    'trend_drift_candidates_path',
    'core_cache_paths',
    'phase2_cache_loaded',
    'lag_paths',
]

for var_name in phase2_temp_vars:
    if var_name in globals( ):
        del globals( )[ var_name ]


## Phase 3 — Feature Engineering
Goal: build the lagged, accumulated features your models will actually use

- 3.1 construct rolling-mean atmospheric features at multiple windows
  - 24hr, 72hr, 7-day, 14-day air temp averages
  - in retrospect, 1, 7 and 28 days
  - but note that these don't really match the lag analysis done in phase 2
  - so that needs decisions,...
- 3.2 construct rate-of-change features (first diffs) for air temp and wind speed
- 3.3 maybe measure time cyclically?
  - day-of-year as (sin, cos) pair
  - hour-of-day similarly if using 1hr data
  - this just so hours 0 and 23, or days 1 and 365 don't SEEM far apart when they're right next to each other
  - some libraries probably do this already, btw...
### NOTE -- Phase 3 feeds both 6 and 7.

### 3.1 - rolling air temps

In [ ]:
# to get rolling averages we dont want air temp NAs
# but we don't want to delete other data... 
# so here's a temp copy
phase3 = water.copy( )
phase3[ 'datetime' ] = pd.to_datetime( phase3[ 'datetime' ], errors = 'coerce' )
phase3 = phase3.dropna( subset = [ 'datetime' ] )
phase3[ 'date' ] = phase3[ 'datetime' ].dt.floor( 'D' )

# now, get the resolution down to daily means and we'll roll them avergages
daily_air = (
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg(
        air_temp = ( 'air_temp', 'mean' ),
        wind_speed = ( 'wind_speed', 'mean' ),
        solar = ( 'solar_radiation', 'mean' ),
        precip = ( 'precipitation', 'sum' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)

# over-engineer a bit and get a whole bunch of rolling averages for different windows and features
for s in [ 'air_temp', 'wind_speed', 'precip', 'solar' ]:
    for w in [ 1, 7, 28 ]: 
        daily_air[ f'{s}_r{w}d' ] = (
            daily_air
            .groupby( [ 'region', 'station' ] )[ s ]
            .transform( lambda s: s.shift( 1 ).rolling( window = w, min_periods = 1 ).mean( ) )
        )

# that shift 1 keeps the current day out of the average

In [ ]:
daily_air.describe().round(3).T

Hmm... some of those mins and maxes look wild

### 3.3 - assessing the day of year as a circle rather than a line

This one will need some reading... it took a fair bit to get it, especially needing both sin and cos

To wit (not hoid):
- real stations peak at different times (phase shifts) of the year (diff climates)
- with both sin and cos, the model can form a(sin) + b(cos), which is a sine wave with any phase offset

In [ ]:
# 0.25 includes leap days
daily_air['doy_sin'] = np.sin( 2 * np.pi * daily_air['date'].dt.dayofyear / 365.25 )
daily_air['doy_cos'] = np.cos( 2 * np.pi * daily_air['date'].dt.dayofyear / 365.25 )

## Phase 4 — Water-Property Delta-From-Mean Features
Goal: construct station-relative anomaly features from baseline-period means

- 4.1 pick a baseline period and compute station baseline means
- 4.2 compute one delta feature example (salinity)

### NOTE -- Phase 4 primarily feeds Phase 7+ modeling.

In [ ]:
baseline_start = '1995-01-01'
baseline_end = '2000-12-31'

# daily salinity + baseline salinity + delta salinity
daily_water = (
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( 
        salinity = ( 'salinity', 'mean' ),
        oxygen = ( 'oxygen', 'mean' ),
        ph = ( 'ph', 'mean' ),
        depth = ( 'depth', 'mean' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)

In [ ]:
properties_baseline = (
    daily_water
    .loc[ daily_water[ 'date' ].between( baseline_start, baseline_end ) ]
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        salinity_baseline = ( 'salinity', 'mean' ),
        oxygen_baseline = ( 'oxygen', 'mean' ),
        ph_baseline = ( 'ph', 'mean' ),
        depth_baseline = ( 'depth', 'mean' )
    )
)

In [ ]:
daily_water = daily_water.merge( 
    properties_baseline,
    on = [ 'region', 'station' ],
    how = 'left',
)

daily_water[ 'delta_salinity' ] = daily_water[ 'salinity' ] - daily_water[ 'salinity_baseline' ]
daily_water[ 'delta_oxygen' ] = daily_water[ 'oxygen' ] - daily_water[ 'oxygen_baseline' ]
daily_water[ 'delta_ph' ] = daily_water[ 'ph' ] - daily_water[ 'ph_baseline' ]
daily_water[ 'delta_depth' ] = daily_water[ 'depth' ] - daily_water[ 'depth_baseline' ]

In [ ]:
daily_water.describe( ).round( 3 ).T

In [ ]:
# phase 4 cleanp .. keep daily_air and daily_water for later modeling phases
phase4_temp_vars = [ 
    'baseline_start',
    'baseline_end',
    'properties_baseline',
    'phase3',
]

for var_name in phase4_temp_vars:
    if var_name in globals( ):
        del globals( )[ var_name ]


## Phase 5 — Regime Transition Detection
Goal: identify natural validation cases and a forward-projection threshold

- 5.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record
- 5.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory
- 5.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?
- 5.4 designate confirmed transitioning stations as a held-out validation set
  - do not use in model training

### NOTE -- Phase 5 must complete before Phase 6

In [ ]:
# start with a copy of daily_air but build out five year rolling averages
fiveyear = daily_air.copy( )
fiveyear[ 'date' ] = pd.to_datetime( fiveyear[ 'date' ], errors = 'coerce' )
fiveyear = fiveyear.dropna( subset = [ 'date' ] )
fiveyear = fiveyear.sort_values( [ 'region', 'station', 'date' ] ).set_index( 'date' )

In [ ]:
fiveyear.describe( ).round( 3 ).T

In [ ]:
fiveyear[ 'water_temp_5yr' ] = (
    fiveyear
    .groupby( [ 'region', 'station' ] )[ 'water_temp' ]
    .transform( lambda s: s.rolling( window = 365 * 5, min_periods = 365 * 3 ).mean( ) )
)

## Phase 6 — Model Development: Air → Water Temperature
Goal: predict ΔT_water given atmospheric forcing history

# also stage 1

- 6.1 establish naive baselines
  - persistence forecast and climatological mean by day-of-year
  - record RMSE and MAE
- 6.2 train ridge/lasso regression using lagged atmospheric features
  - interpret coefficients as a sanity check
- 6.3 train gradient boosted model (XGBoost or LightGBM) on same features
  - compare to linear baseline
- 6.4 use walk-forward temporal cross-validation
  - train on years 1–N, test on N+1
  - since reading suggests standard k-fold doesn't work well with this kind of job
- 6.5 evaluate on held-out transitioning stations from phase 5
  - does the model generalize across space?
- 6.6 select best model, document feature importances
  - lag windows that dominate are a reportable finding

In [ ]:
# code code code


## Phase 7 — Model Development: Water Temp → Water Properties
Goal: predict Δsalinity, ΔDO, ΔpH given ΔT_water and atmospheric context

# also stage 2

- 7.1 for each target variable, repeat baseline 
  - linear → gradient boosted sequence from phase 6
- 7.2 use predicted Δ air temp for water as input
  - keeps the pipeline honest and propagates uncertainty
- 7.3 compare response sensitivity by regime
  - does a +2°C warming suppress dissolved oxy more in warm estuaries than cold ones?
  - prolly a yes ... quantifying it is the contribution
- 7.4 document responses as simple lookup relationships
  - e.g., per-degree sensitivities per regime 
  - these are our core scientific deliverables

In [ ]:
# code is okay, I guess ...

## Phase 8 — Nutrient Models
Goal: predict changes in phosphates, nitrates, chlorophyll given water state

# also stage 3 (if time permits)

- 8.1 use the nutrient-inclusive dataset 
  - is okay that coverage is sparser and document that explicitly
- 8.2 same pipeline/process as phase 7, but input features now include water property predictions from earlier
- 8.3 report model skill honestly 
  - nutrient prediction will be noisier; even identifying dominant drivers is a valid result
  - so far, chlorophyll (despite being most obvious to people as blooms) seemed the weakest response?

In [ ]:
# code

## Phase 9 — Forward Projection
Goal: apply response functions to CMIP6 scenarios

- 9.1 obtain CMIP6 projected Δ air temp for your estuary regions under SSP2-4.5 and SSP5-8.5
  - ESGF portal or pre-downscaled products like NASA NEX-GDDP
  - also ote that there's some evidence that CMIP6 has under-estimated climate damage (by overestimating plant growth)
  - and the new estimate thinks it might be about 11% worse than predicted
  - might factor that in (with a *note?)
- 9.2 feed projected Δ air temp into stage 1 model 
  - get projected Δ water temp by decade
- 9.3 Feed projected Δ water temp into stage 2 response functions 
  - get projected Δ salinity, Δ oxygen, Δ pH
- 9.4 Identify projected regime-crossing dates per station under each scenario 
  - *"station X transitions from cool to warm regime between 2045–2060 under SSP5-8.5"* maybe?
- 9.5 propagate and report uncertainty 
  - at minimum, show scenario spread (SSP2 vs SSP5)
  - if time permits, model confidence intervals

In [ ]:
# code, though...